In [ ]:
FILES={'audit_submission_coverage.py': '"""Stream submission ZIPs to measure structure/coverage, never estimate gold metrics.\n\nMemory is bounded by one query and compact numeric summaries; no corpus is loaded.\n"""\nimport argparse\nfrom collections import Counter\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport statistics\nimport zipfile\n\n\ndef iter_queries(stream, block=65536):\n    decoder = json.JSONDecoder()\n    buf = \'\'\n    eof = False\n    started = False\n    need_separator = False\n    while True:\n        if not eof:\n            more = stream.read(block)\n            eof = not more\n            buf += more\n        buf = buf.lstrip()\n        if not started:\n            if not buf and not eof:\n                continue\n            if not buf.startswith(\'[\'):\n                raise ValueError(\'Expected a JSON array\')\n            buf = buf[1:].lstrip()\n            started = True\n        while buf:\n            if buf[0] == \']\':\n                if (buf[1:] + stream.read()).strip():\n                    raise ValueError(\'Trailing JSON data\')\n                return\n            if need_separator:\n                if buf[0] != \',\':\n                    raise ValueError(\'Missing query separator\')\n                buf = buf[1:].lstrip()\n                need_separator = False\n                if not buf:\n                    break\n            try:\n                item, end = decoder.raw_decode(buf)\n            except json.JSONDecodeError:\n                break\n            if not isinstance(item, dict):\n                raise ValueError(\'Expected query object\')\n            yield item\n            buf = buf[end:].lstrip()\n            need_separator = True\n        if eof:\n            raise ValueError(\'Truncated JSON array\')\n\n\ndef describe(values):\n    values = sorted(values)\n    if not values:\n        return {}\n    return dict(mean=round(statistics.mean(values), 3),\n                p10=values[int((len(values)-1)*.1)],\n                median=statistics.median(values), p90=values[int((len(values)-1)*.9)], max=values[-1])\n\n\ndef audit(path):\n    stats = {k: [] for k in [\'docs\', \'chunks\', \'covered_docs\', \'repeat_windows\', \'chars_per_query\',\n                            \'chars_per_chunk\', \'words_per_chunk\', \'outside_doc_list\', \'duplicate_entries\']}\n    bands = [(0,20),(20,50),(50,100),(100,150),(150,200)]\n    covered, totals = Counter(), Counter()\n    qids, lengths = [], Counter()\n    doc_digest, chunk_digest = hashlib.sha256(), hashlib.sha256()\n    with zipfile.ZipFile(path) as archive:\n        members = [i for i in archive.infolist() if not i.is_dir()]\n        if len(members) != 1 or not members[0].filename.endswith(\'.json\'):\n            raise ValueError(f\'{path}: expected one JSON member\')\n        info = members[0]\n        with archive.open(info) as raw, io.TextIOWrapper(raw, encoding=\'utf-8-sig\') as stream:\n            for entry in iter_queries(stream):\n                qid = int(entry[\'id\'])\n                docs = list(map(str, entry[\'relevant_docs\']))\n                chunks = entry[\'relevant_chunks\']\n                cdocs = {str(c[\'doc_id\']) for c in chunks}\n                texts = [c[\'chunk_text\'] for c in chunks]\n                qids.append(qid)\n                vals = dict(docs=len(docs), chunks=len(chunks), covered_docs=len(cdocs),\n                            repeat_windows=len(chunks)-len(cdocs), chars_per_query=sum(map(len,texts)),\n                            outside_doc_list=len(cdocs-set(docs)),\n                            duplicate_entries=len(chunks)-len({(str(c[\'doc_id\']),c[\'chunk_text\']) for c in chunks}))\n                for k,v in vals.items(): stats[k].append(v)\n                stats[\'chars_per_chunk\'].extend(map(len,texts))\n                stats[\'words_per_chunk\'].extend(len(t.split()) for t in texts)\n                for t in texts:\n                    lengths[\'lt500\'] += len(t)<500\n                    lengths[\'gt6000\'] += len(t)>6000\n                for lo,hi in bands:\n                    key=f\'{lo+1}-{hi}\'\n                    covered[key] += sum(d in cdocs for d in docs[lo:hi])\n                    totals[key] += len(docs[lo:hi])\n                doc_digest.update(json.dumps([qid,sorted(set(docs))],separators=(\',\',\':\')).encode())\n                chunk_digest.update(json.dumps([qid,sorted((str(c[\'doc_id\']),c[\'chunk_text\']) for c in chunks)],\n                                               ensure_ascii=False,separators=(\',\',\':\')).encode())\n    return dict(path=str(path), zip_bytes=path.stat().st_size, json_bytes=info.file_size,\n                compression=info.compress_type, queries=len(qids), unique_queries=len(set(qids)),\n                qids=qids, stats={k:describe(v) for k,v in stats.items()},\n                length_counts=dict(lengths), doc_set_digest=doc_digest.hexdigest(),\n                chunk_set_digest=chunk_digest.hexdigest(),\n                covered_fraction_by_doc_rank={k:round(covered[k]/totals[k],5) if totals[k] else None for k in totals})\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'paths\', nargs=\'+\', type=Path)\n    p.add_argument(\'--output\', required=True, type=Path)\n    args=p.parse_args()\n    out=[]\n    for path in args.paths:\n        result=audit(path)\n        out.append(result)\n        args.output.parent.mkdir(parents=True,exist_ok=True)\n        args.output.write_text(json.dumps(out,ensure_ascii=False,indent=2),encoding=\'utf8\')\n        print(path.name, result[\'queries\'], {k:result[\'stats\'][k][\'mean\'] for k in [\'docs\',\'chunks\',\'covered_docs\']},\n              result[\'covered_fraction_by_doc_rank\'], flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'evidence_spans.py': '"""Contiguous source passages for reranking; offsets always refer to original chunk text.\n\nThese helpers do not change the frozen chunk corpus or embeddings.\n"""\nfrom bisect import bisect_left, bisect_right\nimport re\n\n\ndef join_overlapping(rows):\n    """Return (original start, source text); reject unknown gaps/inconsistent overlaps."""\n    if not rows:\n        raise ValueError(\'No source rows\')\n    rows=sorted(rows,key=lambda r:r[\'start\'])\n    start=rows[0][\'start\'];text=rows[0][\'text\'];end=rows[0][\'end\']\n    sha=rows[0][\'sha\']\n    for row in rows:\n        if row[\'sha\']!=sha or row[\'end\']-row[\'start\']!=len(row[\'text\']):\n            raise ValueError(\'Invalid source identity or offsets\')\n    for row in rows[1:]:\n        if row[\'start\']>end:\n            raise ValueError(\'Unknown source gap; cannot invent separator text\')\n        overlap=min(end,row[\'end\'])-row[\'start\']\n        if text[row[\'start\']-start:row[\'start\']-start+overlap]!=row[\'text\'][:overlap]:\n            raise ValueError(\'Source chunks disagree in overlapping region\')\n        if row[\'end\']>end:\n            text+=row[\'text\'][end-row[\'start\']:];end=row[\'end\']\n    return start,text\n\n\ndef contiguous_around_anchor(rows, anchor):\n    """Keep the maximal known contiguous component containing the seed.\n\n    A distant gap must not discard the seed\'s valid left/right neighbours.\n    Unknown characters across a gap are never reconstructed or guessed.\n    """\n    if not rows or not any(row == anchor for row in rows):\n        raise ValueError(\'Anchor must be one of the source rows\')\n    components=[]\n    for row in sorted(rows,key=lambda r:r[\'start\']):\n        if row[\'sha\'] != anchor[\'sha\']:\n            raise ValueError(\'Invalid source identity\')\n        if not components or row[\'start\'] > max(r[\'end\'] for r in components[-1]):\n            components.append([row])\n        else:\n            components[-1].append(row)\n    # Validate all components, including those outside the selected region.\n    joined=[join_overlapping(component) for component in components]\n    for start,text in joined:\n        if start <= anchor[\'start\'] and start+len(text) >= anchor[\'end\']:\n            if text[anchor[\'start\']-start:anchor[\'end\']-start] != anchor[\'text\']:\n                raise ValueError(\'Anchor disagrees with source component\')\n            return start,text\n    raise ValueError(\'Anchor not covered by source component\')\n\n\ndef centered_span(text, offsets, anchor_start, anchor_end, max_tokens):\n    """A token-budgeted, contiguous source slice around a retrieved seed\'s centre.\n\nUse tokenizer offset_mapping, not decoded tokens (which can rewrite whitespace).\nSnap inward to sentence boundaries only within 10% of the token budget.\n"""\n    if not text or not offsets or max_tokens<1 or not 0<=anchor_start<anchor_end<=len(text):\n        raise ValueError(\'Invalid source/anchor/token budget\')\n    offsets=[(int(a),int(b)) for a,b in offsets if b>a]\n    if not offsets:raise ValueError(\'No nonempty token offsets\')\n    if len(offsets)<=max_tokens:return 0,len(text),text\n    midpoint=(anchor_start+anchor_end)//2\n    centre=max(0,bisect_right([a for a,_ in offsets],midpoint)-1)\n    lo=max(0,min(centre-max_tokens//2,len(offsets)-max_tokens));hi=lo+max_tokens\n    begin=offsets[lo][0];end=offsets[hi-1][1]\n    margin=max(1,max_tokens//10)\n    left_limit=offsets[min(lo+margin,hi-1)][0]\n    right_limit=offsets[max(lo,hi-margin)][0]\n    boundaries=[m.end() for m in re.finditer(r\'(?:[.!?。！？](?:\\s+|(?=[\\u3400-\\u9fff]))|\\n+)\',text[begin:end])]\n    left=[begin+x for x in boundaries if begin+x<=left_limit and begin+x<midpoint]\n    right=[begin+x for x in boundaries if begin+x>=right_limit and begin+x>midpoint]\n    if left:begin=left[0]\n    if right:end=right[-1]\n    assert 0<=begin<end<=len(text) and begin<=midpoint<=end\n    return begin,end,text[begin:end]\n', 'passage_candidates.py': '"""Build source-exact windows for one query, with bounded per-query state."""\nimport hashlib\nimport json\n\nfrom evidence_spans import contiguous_around_anchor, centered_span\n\n\ndef prepare_query(qid, ranked, base_docs, rows, docs_of, tokenizer,\n                  doc_depth=400, passage_tokens=768):\n    best={}\n    for cid in ranked:\n        for doc in docs_of(cid):best.setdefault(int(doc),[]).append(cid)\n    candidates=list(dict.fromkeys(map(int,base_docs)))\n    listed=set(candidates)\n    candidates += [d for d in best if d not in listed]\n    candidates=candidates[:doc_depth]\n    by_pos={(r[\'sha\'],r[\'order\']):cid for cid,r in rows.items()}\n    ranks={cid:i for i,cid in enumerate(ranked)}\n    seen=set();output=[];stats=dict(seed_only=0,context_extended=0,windows=0)\n    for doc in candidates:\n        if doc not in best:continue\n        sha=rows[best[doc][0]][\'sha\']\n        if sha in seen:continue\n        seen.add(sha);seeds=[];spans=set()\n        for cid in best[doc]:\n            if all(abs(rows[cid][\'order\']-rows[c][\'order\'])>1 for c in seeds):seeds.append(cid)\n            if len(seeds)==2:break\n        for cid in seeds:\n            row=rows[cid]\n            neighbours=[rows[by_pos[(sha,o)]] for o in range(row[\'order\']-2,row[\'order\']+3)\n                        if (sha,o) in by_pos]\n            source_start,source=contiguous_around_anchor(neighbours,row)\n            offsets=tokenizer(source,add_special_tokens=False,return_offsets_mapping=True)[\'offset_mapping\']\n            a,b,text=centered_span(source,offsets,row[\'start\']-source_start,row[\'end\']-source_start,passage_tokens)\n            tokens=len(tokenizer.encode(text,add_special_tokens=False))\n            if tokens>passage_tokens:\n                a,b,text=centered_span(source,offsets,row[\'start\']-source_start,row[\'end\']-source_start,passage_tokens-8)\n                tokens=len(tokenizer.encode(text,add_special_tokens=False))\n            assert tokens<=passage_tokens+2 and text==source[a:b]\n            start,end=source_start+a,source_start+b\n            if (start,end) in spans:continue\n            spans.add((start,end))\n            stats[\'windows\']+=1\n            stats[\'seed_only\']+=int(source_start==row[\'start\'] and source==row[\'text\'])\n            stats[\'context_extended\']+=int(len(source)>len(row[\'text\']))\n            text_hash=hashlib.sha256(text.encode()).hexdigest()\n            pid=hashlib.sha256(json.dumps([int(qid),sha,start,end,text_hash],separators=(\',\',\':\')).encode()).hexdigest()\n            output.append(dict(passage_id=pid,query_id=int(qid),doc_id=int(doc),\n                text_sha256=sha,seed=cid,seed_rank=ranks[cid],doc_ids=list(map(int,docs_of(cid))),\n                start=start,end=end,chunk_text=text,passage_tokens=tokens))\n    if not output:raise ValueError(f\'No passages for query {qid}\')\n    return output,stats\n', 'prepare_passage_corpus.py': '"""Kaggle CPU: stream candidate sources into SQLite, then emit resumable query parts.\n\nRAM estimate: <2 GB for Arrow batches, pool, query lists and per-query text; no\nwhole-corpus Python dictionary. SQLite is temporary disk state, not an output.\n"""\nimport argparse\nfrom collections import Counter\nimport gc\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport shutil\nimport sqlite3\nimport time\nimport zipfile\n\nimport pandas as pd\nimport pyarrow as pa\nimport pyarrow.compute as pc\nimport pyarrow.parquet as pq\nfrom transformers import AutoTokenizer\n\nfrom audit_submission_coverage import iter_queries\nfrom passage_candidates import prepare_query\n\n\ndef sha(path):\n    with Path(path).open(\'rb\') as stream:return hashlib.file_digest(stream,\'sha256\').hexdigest()\n\n\ndef one(root,pattern):\n    hits=sorted(root.rglob(pattern))\n    if len(hits)!=1:raise ValueError(f\'{pattern}: expected one input, got {len(hits)}\')\n    return hits[0]\n\n\ndef base_documents(root):\n    hits=list(root.rglob(\'medxir-hybrid-submission.json\'))\n    output={}\n    def consume(stream):\n        for entry in iter_queries(stream):\n            qid=int(entry[\'id\'])\n            if qid in output:raise ValueError(\'Duplicate base query\')\n            output[qid]=list(map(int,entry[\'relevant_docs\']))\n    if len(hits)==1:\n        with hits[0].open(encoding=\'utf-8-sig\') as stream:consume(stream)\n    elif not hits:\n        with zipfile.ZipFile(one(root,\'medxir-hybrid-submission.zip\')) as archive:\n            names=[n for n in archive.namelist() if not n.endswith(\'/\')]\n            if len(names)!=1:raise ValueError(\'Ambiguous base ZIP\')\n            with archive.open(names[0]) as raw,io.TextIOWrapper(raw,encoding=\'utf-8-sig\') as stream:consume(stream)\n    else:raise ValueError(\'Ambiguous base JSON\')\n    return output\n\n\ndef build_source_index(root,out,needed):\n    db_path=out/\'source_cache.sqlite\'\n    # Rebuild after a crash during ingest, never reuse a partial index.\n    if db_path.exists():db_path.unlink()\n    db=sqlite3.connect(db_path)\n    db.execute(\'PRAGMA journal_mode=OFF\');db.execute(\'PRAGMA synchronous=OFF\')\n    db.execute(\'PRAGMA cache_size=-65536\')\n    db.execute(\'CREATE TABLE chunks (cid TEXT PRIMARY KEY, prefix TEXT, sha TEXT, doc INTEGER, ordinal INTEGER, start INTEGER, stop INTEGER, text TEXT, lang TEXT)\')\n    db.execute(\'CREATE TABLE groups (sha TEXT PRIMARY KEY, docs TEXT)\')\n    value_set=pa.array(sorted(needed));files=[];kept=0;source_bytes=0\n    columns=[\'chunk_id\',\'text_sha256\',\'rep_doc_id\',\'order\',\'start\',\'end\',\'chunk_text\']\n    paths=sorted(root.glob(\'**/chunks/*/part-*.parquet\'))\n    if not paths:raise ValueError(\'No chunk corpus parts\')\n    for path in paths:\n        for batch in pq.ParquetFile(path).iter_batches(batch_size=4096,columns=columns):\n            table=pa.Table.from_batches([batch])\n            prefix=pc.utf8_slice_codeunits(table[\'text_sha256\'],0,16)\n            table=table.filter(pc.is_in(prefix,value_set=value_set))\n            values=[]\n            for r in table.to_pylist():\n                if r[\'end\']-r[\'start\']!=len(r[\'chunk_text\']):raise ValueError(\'Bad corpus source offsets\')\n                if r[\'chunk_id\'].rsplit(\'-\',1)[0]!=r[\'text_sha256\'][:16]:raise ValueError(\'Bad chunk identity\')\n                values.append((r[\'chunk_id\'],r[\'text_sha256\'][:16],r[\'text_sha256\'],int(r[\'rep_doc_id\']),\n                    int(r[\'order\']),int(r[\'start\']),int(r[\'end\']),r[\'chunk_text\'],path.parent.name))\n                source_bytes+=len(r[\'chunk_text\'].encode())\n            db.executemany(\'INSERT INTO chunks VALUES (?,?,?,?,?,?,?,?,?)\',values);kept+=len(values)\n        db.commit()\n        files.append(dict(path=str(path.relative_to(root)),rows=pq.ParquetFile(path).metadata.num_rows,sha256=sha(path)))\n        print(\'indexed\',path.parent.name,path.name,\'kept\',kept,flush=True)\n    db.execute(\'CREATE INDEX by_prefix ON chunks(prefix)\')\n    if db.execute(\'SELECT prefix FROM chunks GROUP BY prefix HAVING COUNT(DISTINCT sha)>1 LIMIT 1\').fetchone():\n        raise ValueError(\'SHA prefix collision\')\n    for batch in pq.ParquetFile(one(root,\'hash_groups.parquet\')).iter_batches(batch_size=4096,\n            columns=[\'text_sha256\',\'doc_ids\',\'fanout_safe\']):\n        records=[]\n        for r in batch.to_pylist():\n            if r[\'text_sha256\'][:16] in needed and r[\'fanout_safe\']:\n                ids=list(map(int,r[\'doc_ids\']))\n                if len(ids)!=len(set(ids)):raise ValueError(\'Duplicate fanout ID\')\n                records.append((r[\'text_sha256\'],json.dumps(ids)))\n        db.executemany(\'INSERT INTO groups VALUES (?,?)\',records)\n    db.commit()\n    return db,dict(indexed_chunks=kept,source_text_bytes=source_bytes,source_files=files)\n\n\ndef query_context(db,ranked):\n    prefixes=list(dict.fromkeys(c.rsplit(\'-\',1)[0] for c in ranked))\n    marks=\',\'.join(\'?\' for _ in prefixes)\n    rows={cid:dict(sha=s,doc=d,order=o,start=a,end=b,text=t,lang=l)\n          for cid,p,s,d,o,a,b,t,l in db.execute(f\'SELECT * FROM chunks WHERE prefix IN ({marks})\',prefixes)}\n    if set(ranked)-set(rows):raise ValueError(\'Pool chunk missing in source corpus\')\n    shas=list(dict.fromkeys(r[\'sha\'] for r in rows.values()))\n    groups={s:json.loads(ds) for s,ds in db.execute(\'SELECT * FROM groups WHERE sha IN (\'+\',\'.join(\'?\' for _ in shas)+\')\',shas)}\n    def docs_of(cid):return groups.get(rows[cid][\'sha\']) or [rows[cid][\'doc\']]\n    return rows,docs_of\n\n\ndef main():\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--input\',type=Path,default=Path(\'/kaggle/input\'))\n    parser.add_argument(\'--output\',type=Path,default=Path(\'/kaggle/working\'))\n    parser.add_argument(\'--resume-root\',type=Path)\n    parser.add_argument(\'--max-queries\',type=int,default=1200)\n    args=parser.parse_args();root=args.input;out=args.output;out.mkdir(parents=True,exist_ok=True)\n    started=time.monotonic();pool_path=one(root,\'pool.parquet\')\n    pool=pd.read_parquet(pool_path,columns=[\'query_id\',\'chunk_id\',\'final_rank\']).sort_values([\'query_id\',\'final_rank\'])\n    query_path=one(root,\'query.parquet\')\n    queries=pd.read_parquet(query_path,columns=[\'id\',\'query\'])\n    qids=sorted(map(int,pool.query_id.unique()),key=lambda q:hashlib.sha256(str(q).encode()).hexdigest())[:args.max_queries]\n    if len(qids)!=args.max_queries:raise ValueError(\'Wrong number of pool queries\')\n    if queries.id.duplicated().any() or not set(qids)<=set(queries.id):raise ValueError(\'Query input mismatch\')\n    if args.max_queries==1200 and set(qids)!=set(range(1,1201)):raise ValueError(\'Unexpected full query IDs\')\n    base=base_documents(root)\n    if not set(qids)<=set(base):raise ValueError(\'Base submission missing queries\')\n    pool=pool[pool.query_id.isin(qids)]\n    ranked={int(q):g.chunk_id.tolist() for q,g in pool.groupby(\'query_id\')}\n    for q,ids in ranked.items():\n        if len(ids)!=len(set(ids)):raise ValueError(\'Duplicate pool candidate\')\n    spec=dict(pool_sha256=sha(pool_path),query_sha256=sha(query_path),doc_depth=400,\n              passage_tokens=768,seeds_per_text=2,neighbour_radius=2,\n              source_policy=\'maximal contiguous component; exact substring\',query_ids=qids,\n              implementation_sha256=sha(Path(__file__).with_name(\'passage_candidates.py\')),\n              span_implementation_sha256=sha(Path(__file__).with_name(\'evidence_spans.py\')),\n              tokenizer_config_sha256=sha(one(root,\'bge-reranker-v2-m3/config.json\')))\n    parts_dir=out/\'passage_candidates\';parts_dir.mkdir(exist_ok=True)\n    manifest=dict(complete=False,spec=spec,parts=[],pairs=0,queries=0)\n    manifest_path=out/\'passage_candidates_manifest.json\'\n    if args.resume_root:\n        old=json.loads((args.resume_root/\'passage_candidates_manifest.json\').read_text())\n        if old[\'spec\']!=spec:raise ValueError(\'Resume contract changed\')\n        for part in old[\'parts\']:\n            source=args.resume_root/part[\'path\']\n            if sha(source)!=part[\'sha256\']:raise ValueError(\'Resume part corrupted\')\n            target=out/part[\'path\'];target.parent.mkdir(exist_ok=True)\n            shutil.copyfile(source,target);manifest[\'parts\'].append(part)\n    done={q for part in manifest[\'parts\'] for q in part[\'query_ids\']}\n    needed={c.rsplit(\'-\',1)[0] for q in qids if q not in done for c in ranked[q]}\n    tokenizer=AutoTokenizer.from_pretrained(one(root,\'bge-reranker-v2-m3/config.json\').parent,local_files_only=True)\n    db,index_info=build_source_index(root,out,needed) if needed else (None,{})\n    manifest[\'source_index\']=index_info;totals=Counter();buffer=[];buffer_qids=[]\n    def checkpoint():\n        manifest[\'pairs\']=sum(p[\'pairs\'] for p in manifest[\'parts\'])\n        manifest[\'queries\']=sum(len(p[\'query_ids\']) for p in manifest[\'parts\'])\n        manifest[\'elapsed_seconds\']=time.monotonic()-started\n        temp=manifest_path.with_suffix(\'.tmp\');temp.write_text(json.dumps(manifest,indent=2));temp.replace(manifest_path)\n    def flush():\n        if not buffer:return\n        path=parts_dir/f\'part-{len(manifest["parts"]):03d}.parquet\'\n        frame=pd.DataFrame(buffer)\n        if frame.passage_id.duplicated().any():raise ValueError(\'Duplicate passage ID\')\n        frame.to_parquet(path,index=False,compression=\'zstd\')\n        manifest[\'parts\'].append(dict(path=str(path.relative_to(out)),sha256=sha(path),\n                                     pairs=len(frame),query_ids=list(buffer_qids)))\n        buffer.clear();buffer_qids.clear();checkpoint()\n    checkpoint()\n    for qid in qids:\n        if qid in done:continue\n        rows,docs_of=query_context(db,ranked[qid])\n        current,stats=prepare_query(qid,ranked[qid],base[qid],rows,docs_of,tokenizer)\n        totals.update(stats);buffer.extend(current);buffer_qids.append(qid)\n        print(\'prepared\',qid,\'pairs\',len(current),\'context_extended\',stats[\'context_extended\'],flush=True)\n        del rows,docs_of,current\n        if len(buffer_qids)>=32:flush();gc.collect()\n    flush()\n    if db:db.close();(out/\'source_cache.sqlite\').unlink()\n    queries[queries.id.isin(qids)].to_parquet(out/\'passage_queries.parquet\',index=False)\n    manifest.update(complete=True,diagnostics=dict(totals),query_file_sha256=sha(out/\'passage_queries.parquet\'))\n    checkpoint()\n    assert manifest[\'queries\']==len(qids)\n    print(\'COMPLETE\',manifest[\'queries\'],manifest[\'pairs\'],\'seconds\',round(manifest[\'elapsed_seconds\']),flush=True)\n\n\nif __name__==\'__main__\':main()\n'}
ARGS=[]
from pathlib import Path
import sys,subprocess
for name,source in FILES.items():Path('/kaggle/working',name).write_text(source,encoding='utf8')
subprocess.run([sys.executable,'-u','/kaggle/working/prepare_passage_corpus.py',*ARGS],check=True)
